In [ ]:
# FlashInfer go/no-go gate, step 1: install, import, confirm the standalone decode API
!pip -q install flashinfer-python 2>&1 | tail -5

import torch, math, time
print("torch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

try:
    import flashinfer
    print("flashinfer import OK, version:", getattr(flashinfer, "__version__", "unknown"))
    print("has single_decode_with_kv_cache:", hasattr(flashinfer, "single_decode_with_kv_cache"))
    # list decode-related callables so we see the real API surface
    print("decode-related attrs:", [a for a in dir(flashinfer) if "decode" in a.lower()])
except Exception as ex:
    import traceback; traceback.print_exc()
    print("\nIMPORT FAILED — paste this; that's the no-go signal on install.")

torch: 2.11.0+cu130 | CUDA: 13.0
GPU: NVIDIA A100-SXM4-40GB
flashinfer import OK, version: 0.7.0.post1
has single_decode_with_kv_cache: True
decode-related attrs: ['BatchDecodeWithPagedKVCacheWrapper', 'BatchDecodeWithSharedPrefixPagedKVCacheWrapper', 'CUDAGraphBatchDecodeWithPagedKVCacheWrapper', 'cudnn_batch_decode_with_kv_cache', 'decode', 'fast_decode_plan', 'fused_kda_decode', 'gdn_fused_decode_step', 'gdn_fused_decode_step_supported', 'kda_decode', 'packed_kda_decode', 'single_decode_with_kv_cache', 'sm110_gqa_decode']


In [ ]:
# go/no-go step 2: one single_decode call at head_dim=128, verified against PyTorch SDPA
import torch, math, torch.nn.functional as F
dev = "cuda"; torch.manual_seed(0)

N, H, Hkv, D = 4096, 32, 8, 128   # kv_len, q_heads, kv_heads, head_dim (FlashInfer happy path)

# FlashInfer single_decode signature: q [num_qo_heads, head_dim], k/v [kv_len, num_kv_heads, head_dim]
q = torch.randn(H, D, dtype=torch.float16, device=dev)
k = torch.randn(N, Hkv, D, dtype=torch.float16, device=dev)
v = torch.randn(N, Hkv, D, dtype=torch.float16, device=dev)

try:
    o_fi = flashinfer.single_decode_with_kv_cache(q, k, v)   # [H, D]
    print("flashinfer decode call OK, out shape:", tuple(o_fi.shape))

    # reference: SDPA with GQA expansion
    G = H // Hkv
    kE = k.transpose(0,1).repeat_interleave(G, dim=0)   # [H, N, D]
    vE = v.transpose(0,1).repeat_interleave(G, dim=0)
    o_ref = F.scaled_dot_product_attention(
        q.view(H,1,1,D), kE.view(H,1,N,D), vE.view(H,1,N,D)).view(H, D)

    diff = (o_fi.float() - o_ref.float()).abs().max().item()
    rel  = diff / o_ref.float().abs().max().item()
    print(f"flashinfer vs SDPA: max abs {diff:.3e} | rel {rel:.3e}")
    print("CORRECT (matches SDPA)" if torch.allclose(o_fi.float(), o_ref.float(), atol=2e-2)
          else "MISMATCH — API assumption wrong, do not benchmark yet")
except Exception as ex:
    import traceback; traceback.print_exc()
    print("\nDECODE CALL FAILED — paste this. Likely a signature/layout difference we fix before proceeding.")

2026-10-03 05:49:23,920 - INFO - core.py:564 - flashinfer.jit: Building JIT module single_decode_with_kv_cache_dtype_q_f16_dtype_kv_f16_dtype_o_f16_head_dim_qk_128_head_dim_vo_128_posenc_0_use_swa_False_use_logits_cap_False; this can take several minutes on first use.


flashinfer decode call OK, out shape: (32, 128)
flashinfer vs SDPA: max abs 6.104e-05 | rel 6.748e-04
CORRECT (matches SDPA)


In [ ]:
import os, torch
from torch.utils.cpp_extension import load
os.makedirs("kernels", exist_ok=True)
fp16_src = r"""
#include <torch/extension.h>
#include <c10/cuda/CUDAException.h>
#include <cuda_fp16.h>
#include <cuda_runtime.h>
template <int D, int W>
__global__ void decode_attn_fp16_mw_kernel(
    const __half* __restrict__ q, const __half* __restrict__ K, const __half* __restrict__ V,
    __half* __restrict__ o, int N, float scale){
    const int head=blockIdx.x, lane=threadIdx.x&31, warp=threadIdx.x>>5, ELEMS=D/32;
    const __half* qh=q+(size_t)head*D; const __half* Kh=K+(size_t)head*N*D;
    const __half* Vh=V+(size_t)head*N*D; __half* oh=o+(size_t)head*D;
    float q_reg[ELEMS];
    #pragma unroll
    for(int e=0;e<ELEMS;++e) q_reg[e]=__half2float(qh[lane+e*32]);
    float m_run=-INFINITY,l_run=0.0f,o_run[ELEMS];
    #pragma unroll
    for(int e=0;e<ELEMS;++e) o_run[e]=0.0f;
    for(int i=warp;i<N;i+=W){
        const __half* Ki=Kh+(size_t)i*D; float partial=0.0f;
        #pragma unroll
        for(int e=0;e<ELEMS;++e) partial+=q_reg[e]*__half2float(Ki[lane+e*32]);
        #pragma unroll
        for(int off=16;off>0;off>>=1) partial+=__shfl_down_sync(0xffffffff,partial,off);
        float s=__shfl_sync(0xffffffff,partial,0)*scale;
        float m_new=fmaxf(m_run,s), beta=__expf(m_run-m_new), p=__expf(s-m_new);
        l_run=beta*l_run+p; const __half* Vi=Vh+(size_t)i*D;
        #pragma unroll
        for(int e=0;e<ELEMS;++e) o_run[e]=beta*o_run[e]+p*__half2float(Vi[lane+e*32]);
        m_run=m_new;
    }
    __shared__ float sm[W],sl[W],so[W][D];
    if(lane==0){sm[warp]=m_run;sl[warp]=l_run;}
    #pragma unroll
    for(int e=0;e<ELEMS;++e) so[warp][lane+e*32]=o_run[e];
    __syncthreads();
    if(warp==0){
        float m_c=-INFINITY,l_c=0.0f,o_c[ELEMS];
        #pragma unroll
        for(int e=0;e<ELEMS;++e) o_c[e]=0.0f;
        #pragma unroll
        for(int w=0;w<W;++w){ float m_w=sm[w],m_new=fmaxf(m_c,m_w);
            float a=__expf(m_c-m_new),b=__expf(m_w-m_new); l_c=a*l_c+b*sl[w];
            #pragma unroll
            for(int e=0;e<ELEMS;++e) o_c[e]=a*o_c[e]+b*so[w][lane+e*32]; m_c=m_new; }
        #pragma unroll
        for(int e=0;e<ELEMS;++e) oh[lane+e*32]=__float2half(o_c[e]/l_c);
    }
}
torch::Tensor decode_attn_fp16_mw(torch::Tensor q, torch::Tensor K, torch::Tensor V, int64_t W){
    const int B=K.size(0),N=K.size(1),D=K.size(2);
    auto o=torch::empty({B,D},q.options()); const float scale=1.0f/sqrtf((float)D);
    dim3 grid(B),block(32*W);
    #define L(Dv,Wv) decode_attn_fp16_mw_kernel<Dv,Wv><<<grid,block>>>((const __half*)q.data_ptr<at::Half>(),(const __half*)K.data_ptr<at::Half>(),(const __half*)V.data_ptr<at::Half>(),(__half*)o.data_ptr<at::Half>(),N,scale)
    TORCH_CHECK(D==64||D==128,"D");
    if(D==64){switch(W){case 2:L(64,2);break;case 4:L(64,4);break;case 8:L(64,8);break;default:TORCH_CHECK(false,"W");}}
    else{switch(W){case 2:L(128,2);break;case 4:L(128,4);break;case 8:L(128,8);break;default:TORCH_CHECK(false,"W");}}
    #undef L
    TORCH_CHECK(cudaGetLastError()==cudaSuccess,"launch"); return o;
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME,m){m.def("decode_attn_fp16_mw",&decode_attn_fp16_mw,"fp16 fused");}
"""
open("kernels/decode_attn_fp16_mw.cu","w").write(fp16_src)
mod_fp16 = load(name="decode_attn_fp16_mw", sources=["kernels/decode_attn_fp16_mw.cu"], extra_cuda_cflags=["-O3"], verbose=False)
print("mod_fp16 ready")

mod_fp16 ready


In [ ]:
import torch, math, torch.nn.functional as F, flashinfer
dev="cuda"; torch.manual_seed(0)

def bench(fn, iters=100, warmup=30):
    warm=torch.randn(2048,2048,device=dev,dtype=torch.float16)
    for _ in range(30): _=warm@warm
    torch.cuda.synchronize()
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    s,e=torch.cuda.Event(enable_timing=True),torch.cuda.Event(enable_timing=True); ts=[]
    for _ in range(iters):
        s.record(); fn(); e.record(); torch.cuda.synchronize(); ts.append(s.elapsed_time(e))
    ts.sort(); return ts[len(ts)//2]

def compare(N, D, H=32, Hkv=8, W=8):
    G=H//Hkv
    # our kernel wants [B,D] q and [B,N,D] K/V with B = heads, GQA expanded
    q_h   = torch.randn(H, D, dtype=torch.float16, device=dev)
    k_kv  = torch.randn(N, Hkv, D, dtype=torch.float16, device=dev)   # FlashInfer layout
    v_kv  = torch.randn(N, Hkv, D, dtype=torch.float16, device=dev)
    # expanded views for our kernel + SDPA
    kE = k_kv.transpose(0,1).repeat_interleave(G,0).contiguous()      # [H,N,D]
    vE = v_kv.transpose(0,1).repeat_interleave(G,0).contiguous()
    q_b = q_h.contiguous()                                            # [H,D] = [B,D]

    # --- correctness: both vs SDPA ---
    o_ref = F.scaled_dot_product_attention(q_h.view(H,1,1,D), kE.view(H,1,N,D), vE.view(H,1,N,D)).view(H,D)
    o_fi  = flashinfer.single_decode_with_kv_cache(q_h, k_kv, v_kv)
    o_ours= mod_fp16.decode_attn_fp16_mw(q_b, kE, vE, W)
    ok_fi  = torch.allclose(o_fi.float(),  o_ref.float(), atol=2e-2)
    ok_ours= torch.allclose(o_ours.float(),o_ref.float(), atol=2e-2)
    if not (ok_fi and ok_ours):
        print(f"D={D} N={N}: CORRECTNESS FAIL (fi={ok_fi}, ours={ok_ours}) — skipping timing")
        return None
    # --- timing (only after correctness passes) ---
    t_fi   = bench(lambda: flashinfer.single_decode_with_kv_cache(q_h, k_kv, v_kv))
    t_ours = bench(lambda: mod_fp16.decode_attn_fp16_mw(q_b, kE, vE, W))
    print(f"D={D:3d} N={N:5d} | ours {t_ours:.4f} ms | flashinfer {t_fi:.4f} ms | ours/fi {t_fi/t_ours:.2f}x")
    return dict(D=D,N=N,ours_ms=t_ours,flashinfer_ms=t_fi,ratio=t_fi/t_ours)

print("head_dim=128 (FlashInfer's tuned path):")
for N in [1024, 4096, 16384]:
    try: compare(N, 128)
    except Exception as ex: print(f"D=128 N={N} ERROR:", str(ex)[:120])

print("\nhead_dim=64 (Llama-3.2-1B actual shape):")
for N in [1024, 4096, 16384]:
    try: compare(N, 64)
    except Exception as ex: print(f"D=64 N={N} ERROR:", str(ex)[:120])

head_dim=128 (FlashInfer's tuned path):
D=128 N= 1024 | ours 0.0932 ms | flashinfer 0.0543 ms | ours/fi 0.58x
D=128 N= 4096 | ours 0.6636 ms | flashinfer 0.0614 ms | ours/fi 0.09x
D=128 N=16384 | ours 2.2610 ms | flashinfer 0.0922 ms | ours/fi 0.04x

head_dim=64 (Llama-3.2-1B actual shape):
D= 64 N= 1024 | ours 0.0553 ms | flashinfer 0.0471 ms | ours/fi 0.85x
D= 64 N= 4096 | ours 0.4168 ms | flashinfer 0.0461 ms | ours/fi 0.11x
D= 64 N=16384 | ours 1.6497 ms | flashinfer 0.0584 ms | ours/fi 0.04x


In [ ]:
import torch, math
torch.manual_seed(0)
d, N, S = 64, 4096, 8        # head_dim, cached tokens, number of splits along N

q = torch.randn(d,    dtype=torch.float32)
K = torch.randn(N, d, dtype=torch.float32)
V = torch.randn(N, d, dtype=torch.float32)
scale = 1.0 / math.sqrt(d)

# ground truth: single-pass stable attention
scores = (K @ q) * scale
truth  = torch.softmax(scores, 0) @ V

# ---- Stage 1 of split-K: each split does online-softmax over a CONTIGUOUS chunk ----
# (contiguous chunks, not strided: block s owns tokens [s*chunk : (s+1)*chunk])
def partial_online(idx_lo, idx_hi):
    m = torch.tensor(float('-inf')); l = torch.tensor(0.0); o = torch.zeros(d)
    for i in range(idx_lo, idx_hi):
        s = torch.dot(K[i], q) * scale
        m_new = torch.maximum(m, s)
        beta  = torch.exp(m - m_new)
        p     = torch.exp(s - m_new)
        l = beta*l + p
        o = beta*o + p*V[i]
        m = m_new
    return m, l, o

chunk = (N + S - 1) // S
partials = []
for s in range(S):
    lo, hi = s*chunk, min((s+1)*chunk, N)
    if lo >= hi:                       # empty split (N not divisible by S) -> identity partial
        partials.append((torch.tensor(float('-inf')), torch.tensor(0.0), torch.zeros(d)))
    else:
        partials.append(partial_online(lo, hi))

# ---- Stage 2 of split-K: combine the S partials (cross-block online-softmax merge) ----
m_c = torch.tensor(float('-inf')); l_c = torch.tensor(0.0); o_c = torch.zeros(d)
for (m_s, l_s, o_s) in partials:
    m_new = torch.maximum(m_c, m_s)
    a = torch.exp(m_c - m_new)        # rescale accumulated
    b = torch.exp(m_s - m_new)        # rescale incoming partial
    l_c = a*l_c + b*l_s
    o_c = a*o_c + b*o_s
    m_c = m_new
combined = o_c / l_c

err = (combined - truth).abs().max().item()
print(f"split-K (S={S}) vs single-pass truth: max abs diff {err:.3e}")
assert torch.allclose(combined, truth, atol=1e-5), "SPLIT-K COMBINE FAILED"
print("split-K combine identity holds  OK")

# also check an awkward split count (N not divisible by S) to stress empty/partial chunks
for S_test in [7, 13, 100, N+5]:
    chunk = (N + S_test - 1)//S_test
    ps=[]
    for s in range(S_test):
        lo,hi=s*chunk,min((s+1)*chunk,N)
        ps.append(partial_online(lo,hi) if lo<hi else (torch.tensor(float('-inf')),torch.tensor(0.0),torch.zeros(d)))
    mc=torch.tensor(float('-inf'));lc=torch.tensor(0.0);oc=torch.zeros(d)
    for (ms,ls,os_) in ps:
        mn=torch.maximum(mc,ms); a=torch.exp(mc-mn); b=torch.exp(ms-mn)
        lc=a*lc+b*ls; oc=a*oc+b*os_; mc=mn
    e=(oc/lc - truth).abs().max().item()
    print(f"  S={S_test:5d}: diff {e:.3e} {'OK' if e<1e-4 else 'FAIL'}")

split-K (S=8) vs single-pass truth: max abs diff 2.608e-08
split-K combine identity holds  OK
  S=    7: diff 4.098e-08 OK
  S=   13: diff 3.004e-08 OK
  S=  100: diff 2.608e-08 OK
  S= 4101: diff 7.451e-08 OK


In [ ]:
import os, math, torch, torch.nn.functional as F
from torch.utils.cpp_extension import load
os.makedirs("kernels", exist_ok=True)

splitk_src = r'''
#include <torch/extension.h>
#include <c10/cuda/CUDAException.h>
#include <cuda_fp16.h>
#include <cuda_runtime.h>

// ---- PARTIAL kernel: grid=(H, S), block = W warps. Block (h,s) does online-softmax
//      over its contiguous N-chunk and writes (m,l,o) partials to fp32 scratch. ----
template <int D, int W>
__global__ void splitk_partial_kernel(
    const __half* __restrict__ q,   // [H, D]
    const __half* __restrict__ K,   // [H, N, D]
    const __half* __restrict__ V,   // [H, N, D]
    float* __restrict__ m_part,     // [H, S]
    float* __restrict__ l_part,     // [H, S]
    float* __restrict__ o_part,     // [H, S, D]
    int N, int S, float scale)
{
    const int head = blockIdx.x;
    const int split = blockIdx.y;
    const int lane = threadIdx.x & 31, warp = threadIdx.x >> 5, ELEMS = D/32;

    const int chunk = (N + S - 1) / S;
    const int lo = split * chunk;
    const int hi = min(lo + chunk, N);

    const __half* qh = q + (size_t)head*D;
    const __half* Kh = K + (size_t)head*N*D;
    const __half* Vh = V + (size_t)head*N*D;

    float q_reg[ELEMS];
    #pragma unroll
    for(int e=0;e<ELEMS;++e) q_reg[e]=__half2float(qh[lane+e*32]);
    float m_run=-INFINITY, l_run=0.0f, o_run[ELEMS];
    #pragma unroll
    for(int e=0;e<ELEMS;++e) o_run[e]=0.0f;

    // stream this split's chunk, strided by warp within the chunk
    for(int i=lo+warp; i<hi; i+=W){
        const __half* Ki=Kh+(size_t)i*D; float partial=0.0f;
        #pragma unroll
        for(int e=0;e<ELEMS;++e) partial+=q_reg[e]*__half2float(Ki[lane+e*32]);
        #pragma unroll
        for(int off=16;off>0;off>>=1) partial+=__shfl_down_sync(0xffffffff,partial,off);
        float s=__shfl_sync(0xffffffff,partial,0)*scale;
        float m_new=fmaxf(m_run,s), beta=__expf(m_run-m_new), p=__expf(s-m_new);
        l_run=beta*l_run+p; const __half* Vi=Vh+(size_t)i*D;
        #pragma unroll
        for(int e=0;e<ELEMS;++e) o_run[e]=beta*o_run[e]+p*__half2float(Vi[lane+e*32]);
        m_run=m_new;
    }
    // combine the W warps' partials within this block (shared-mem, like before)
    __shared__ float sm[W],sl[W],so[W][D];
    if(lane==0){sm[warp]=m_run;sl[warp]=l_run;}
    #pragma unroll
    for(int e=0;e<ELEMS;++e) so[warp][lane+e*32]=o_run[e];
    __syncthreads();
    if(warp==0){
        float m_c=-INFINITY,l_c=0.0f,o_c[ELEMS];
        #pragma unroll
        for(int e=0;e<ELEMS;++e) o_c[e]=0.0f;
        #pragma unroll
        for(int w=0;w<W;++w){ float mw=sm[w],mn=fmaxf(m_c,mw);
            float a=__expf(m_c-mn),b=__expf(mw-mn); l_c=a*l_c+b*sl[w];
            #pragma unroll
            for(int e=0;e<ELEMS;++e) o_c[e]=a*o_c[e]+b*so[w][lane+e*32]; m_c=mn; }
        // write UNNORMALIZED partial (m, l, o) for this (head, split)
        const size_t base = ((size_t)head*S + split);
        if(lane==0){ m_part[base]=m_c; l_part[base]=l_c; }
        #pragma unroll
        for(int e=0;e<ELEMS;++e) o_part[base*D + lane + e*32] = o_c[e];
    }
}

// ---- COMBINE kernel: grid=H, one warp. Merge S partials -> final output. ----
template <int D>
__global__ void splitk_combine_kernel(
    const float* __restrict__ m_part,  // [H,S]
    const float* __restrict__ l_part,  // [H,S]
    const float* __restrict__ o_part,  // [H,S,D]
    __half* __restrict__ o,            // [H,D]
    int S)
{
    const int head=blockIdx.x, lane=threadIdx.x&31, ELEMS=D/32;
    float m_c=-INFINITY,l_c=0.0f,o_c[ELEMS];
    #pragma unroll
    for(int e=0;e<ELEMS;++e) o_c[e]=0.0f;
    for(int s=0;s<S;++s){
        const size_t base=((size_t)head*S + s);
        float m_s=m_part[base], l_s=l_part[base];
        float mn=fmaxf(m_c,m_s);
        float a=__expf(m_c-mn), b=__expf(m_s-mn);
        l_c=a*l_c+b*l_s;
        #pragma unroll
        for(int e=0;e<ELEMS;++e) o_c[e]=a*o_c[e]+b*o_part[base*D + lane + e*32];
        m_c=mn;
    }
    __half* oh=o+(size_t)head*D;
    #pragma unroll
    for(int e=0;e<ELEMS;++e) oh[lane+e*32]=__float2half(o_c[e]/l_c);
}

torch::Tensor decode_attn_splitk(torch::Tensor q, torch::Tensor K, torch::Tensor V, int64_t S, int64_t W){
    const int H=K.size(0), N=K.size(1), D=K.size(2);
    TORCH_CHECK(D==64||D==128,"D");
    auto fopt = torch::TensorOptions().dtype(torch::kFloat32).device(q.device());
    auto m_part = torch::empty({H,S}, fopt);
    auto l_part = torch::empty({H,S}, fopt);
    auto o_part = torch::empty({H,S,D}, fopt);
    auto o = torch::empty({H,D}, q.options());
    const float scale = 1.0f/sqrtf((float)D);

    dim3 pgrid(H, S), pblock(32*W);
    #define LP(Dv,Wv) splitk_partial_kernel<Dv,Wv><<<pgrid,pblock>>>( \
        (const __half*)q.data_ptr<at::Half>(),(const __half*)K.data_ptr<at::Half>(),(const __half*)V.data_ptr<at::Half>(), \
        m_part.data_ptr<float>(),l_part.data_ptr<float>(),o_part.data_ptr<float>(),N,(int)S,scale)
    if(D==64){switch(W){case 2:LP(64,2);break;case 4:LP(64,4);break;case 8:LP(64,8);break;default:TORCH_CHECK(false,"W");}}
    else{switch(W){case 2:LP(128,2);break;case 4:LP(128,4);break;case 8:LP(128,8);break;default:TORCH_CHECK(false,"W");}}
    #undef LP
    TORCH_CHECK(cudaGetLastError()==cudaSuccess,"partial launch");

    dim3 cgrid(H), cblock(32);
    if(D==64) splitk_combine_kernel<64><<<cgrid,cblock>>>(m_part.data_ptr<float>(),l_part.data_ptr<float>(),o_part.data_ptr<float>(),(__half*)o.data_ptr<at::Half>(),(int)S);
    else      splitk_combine_kernel<128><<<cgrid,cblock>>>(m_part.data_ptr<float>(),l_part.data_ptr<float>(),o_part.data_ptr<float>(),(__half*)o.data_ptr<at::Half>(),(int)S);
    TORCH_CHECK(cudaGetLastError()==cudaSuccess,"combine launch");
    return o;
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME,m){m.def("decode_attn_splitk",&decode_attn_splitk,"split-K decode attention");}
'''
open("kernels/decode_attn_splitk.cu","w").write(splitk_src)
mod_sk = load(name="decode_attn_splitk", sources=["kernels/decode_attn_splitk.cu"], extra_cuda_cflags=["-O3"], verbose=True)
print("split-K compiled OK")

split-K compiled OK


In [ ]:
dev="cuda"; torch.manual_seed(0)
def sdpa_ref(q,K,V):
    H,D=q.shape; N=K.shape[1]
    return F.scaled_dot_product_attention(q.view(H,1,1,D),K.view(H,1,N,D),V.view(H,1,N,D)).view(H,D)

def check_sk(H,N,D,S,W,atol=2e-2):
    q=torch.randn(H,D,dtype=torch.float16,device=dev).contiguous()
    K=torch.randn(H,N,D,dtype=torch.float16,device=dev).contiguous()
    V=torch.randn(H,N,D,dtype=torch.float16,device=dev).contiguous()
    out=mod_sk.decode_attn_splitk(q,K,V,S,W)
    ref=sdpa_ref(q,K,V)
    d=(out-ref).abs().max().item(); ok=torch.allclose(out,ref,atol=atol)
    print(f"D={D:3d} H={H:3d} N={N:6d} S={S:4d} W={W} | diff {d:.3e} | {'PASS' if ok else 'FAIL'}")
    return ok

print("split-K correctness:")
ok=True
for D in [64,128]:
    for W in [4,8]:
        for (N,S) in [(1024,4),(4096,8),(4096,32),(16384,32),(16384,128)]:
            ok&=check_sk(32,N,D,S,W)
    # edge cases: S larger than N (empty splits), S not dividing N, tiny N
    ok&=check_sk(8,3,D,8,4)      # N<S: most splits empty
    ok&=check_sk(8,100,D,7,4)    # N not divisible by S
    ok&=check_sk(8,1,D,4,2)      # N=1
print("\nALL PASS" if ok else "\nSOME FAILED — stop, do not benchmark"); assert ok

split-K correctness:
D= 64 H= 32 N=  1024 S=   4 W=4 | diff 1.221e-04 | PASS
D= 64 H= 32 N=  4096 S=   8 W=4 | diff 6.104e-05 | PASS
D= 64 H= 32 N=  4096 S=  32 W=4 | diff 6.104e-05 | PASS
D= 64 H= 32 N= 16384 S=  32 W=4 | diff 3.052e-05 | PASS
D= 64 H= 32 N= 16384 S= 128 W=4 | diff 3.052e-05 | PASS
D= 64 H= 32 N=  1024 S=   4 W=8 | diff 1.221e-04 | PASS
D= 64 H= 32 N=  4096 S=   8 W=8 | diff 6.104e-05 | PASS
D= 64 H= 32 N=  4096 S=  32 W=8 | diff 6.104e-05 | PASS
D= 64 H= 32 N= 16384 S=  32 W=8 | diff 3.052e-05 | PASS
D= 64 H= 32 N= 16384 S= 128 W=8 | diff 3.052e-05 | PASS
D= 64 H=  8 N=     3 S=   8 W=4 | diff 9.766e-04 | PASS
D= 64 H=  8 N=   100 S=   7 W=4 | diff 2.441e-04 | PASS
D= 64 H=  8 N=     1 S=   4 W=2 | diff 0.000e+00 | PASS
D=128 H= 32 N=  1024 S=   4 W=4 | diff 1.221e-04 | PASS
D=128 H= 32 N=  4096 S=   8 W=4 | diff 6.104e-05 | PASS
D=128 H= 32 N=  4096 S=  32 W=4 | diff 6.104e-05 | PASS
D=128 H= 32 N= 16384 S=  32 W=4 | diff 3.052e-05 | PASS
D=128 H= 32 N= 16384 S= 128

In [ ]:
src = open("kernels/decode_attn_splitk.cu").read()

src = src.replace(
    "for(int w=0;w<W;++w){ float mw=sm[w],mn=fmaxf(m_c,mw);\n            float a=__expf(m_c-mn),b=__expf(mw-mn); l_c=a*l_c+b*sl[w];",
    "for(int w=0;w<W;++w){ float mw=sm[w],mn=fmaxf(m_c,mw);\n            float a=(mn==-INFINITY)?0.0f:__expf(m_c-mn);\n            float b=(mn==-INFINITY)?0.0f:__expf(mw-mn); l_c=a*l_c+b*sl[w];")

src = src.replace(
    "float mn=fmaxf(m_c,m_s);\n        float a=__expf(m_c-mn), b=__expf(m_s-mn);\n        l_c=a*l_c+b*l_s;",
    "float mn=fmaxf(m_c,m_s);\n        float a=(mn==-INFINITY)?0.0f:__expf(m_c-mn);\n        float b=(mn==-INFINITY)?0.0f:__expf(m_s-mn);\n        l_c=a*l_c+b*l_s;")

open("kernels/decode_attn_splitk.cu","w").write(src)
assert src.count("(mn==-INFINITY)?0.0f") == 4, f"expected 4 guards, found {src.count('(mn==-INFINITY)?0.0f')} — paste this, patch missed"
print("patched. recompiling...")

import torch
from torch.utils.cpp_extension import load
mod_sk = load(name="decode_attn_splitk", sources=["kernels/decode_attn_splitk.cu"], extra_cuda_cflags=["-O3"], verbose=False)
print("recompiled OK — now re-run your Cell 2 correctness check")

patched. recompiling...
recompiled OK — now re-run your Cell 2 correctness check


In [ ]:
import torch, math, torch.nn.functional as F, flashinfer
dev="cuda"; torch.manual_seed(0)

def bench(fn, iters=100, warmup=30):
    warm=torch.randn(2048,2048,device=dev,dtype=torch.float16)
    for _ in range(30): _=warm@warm
    torch.cuda.synchronize()
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    s,e=torch.cuda.Event(enable_timing=True),torch.cuda.Event(enable_timing=True); ts=[]
    for _ in range(iters):
        s.record(); fn(); e.record(); torch.cuda.synchronize(); ts.append(s.elapsed_time(e))
    ts.sort(); return ts[len(ts)//2]

# ---- Part 1: sweep S to find the best split count at the shapes that hurt before ----
print("S sweep (D=64, H=32, W=8) — find best split count:")
for N in [1024, 4096, 16384]:
    q=torch.randn(32,64,dtype=torch.float16,device=dev).contiguous()
    K=torch.randn(32,N,64,dtype=torch.float16,device=dev).contiguous()
    V=torch.randn(32,N,64,dtype=torch.float16,device=dev).contiguous()
    row=f"  N={N:6d}: "
    best=(1e9,None)
    for S in [1,4,8,16,32,64,128]:
        t=bench(lambda: mod_sk.decode_attn_splitk(q,K,V,S,8))
        row+=f"S{S}={t:.4f} "
        if t<best[0]: best=(t,S)
    print(row+f" | best S={best[1]} ({best[0]:.4f} ms)")

# ---- Part 2: FlashInfer rematch at the best S, correctness-gated, both head dims ----
print("\nFlashInfer rematch (split-K best-S vs flashinfer vs old single-block):")
def rematch(N, D, H=32, Hkv=8, W=8, S=32):
    G=H//Hkv
    q_h=torch.randn(H,D,dtype=torch.float16,device=dev)
    k_kv=torch.randn(N,Hkv,D,dtype=torch.float16,device=dev)
    v_kv=torch.randn(N,Hkv,D,dtype=torch.float16,device=dev)
    kE=k_kv.transpose(0,1).repeat_interleave(G,0).contiguous()
    vE=v_kv.transpose(0,1).repeat_interleave(G,0).contiguous()
    # correctness: split-K vs SDPA
    o_ref=F.scaled_dot_product_attention(q_h.view(H,1,1,D),kE.view(H,1,N,D),vE.view(H,1,N,D)).view(H,D)
    o_sk=mod_sk.decode_attn_splitk(q_h.contiguous(),kE,vE,S,W)
    if not torch.allclose(o_sk.float(),o_ref.float(),atol=2e-2):
        print(f"  D={D} N={N}: SPLIT-K CORRECTNESS FAIL — skip"); return
    t_sk=bench(lambda: mod_sk.decode_attn_splitk(q_h.contiguous(),kE,vE,S,W))
    t_fi=bench(lambda: flashinfer.single_decode_with_kv_cache(q_h,k_kv,v_kv))
    print(f"  D={D:3d} N={N:6d} S={S:3d} | split-K {t_sk:.4f} ms | flashinfer {t_fi:.4f} ms | sk/fi {t_fi/t_sk:.2f}x")

for D in [64,128]:
    for N in [1024,4096,16384]:
        rematch(N,D,S=32)

S sweep (D=64, H=32, W=8) — find best split count:
  N=  1024: S1=0.0788 S4=0.0379 S8=0.0317 S16=0.0317 S32=0.0369 S64=0.0440 S128=0.0635  | best S=8 (0.0317 ms)
  N=  4096: S1=0.5110 S4=0.1300 S8=0.0799 S16=0.0563 S32=0.0604 S64=0.0625 S128=0.0799  | best S=16 (0.0563 ms)
  N= 16384: S1=1.6916 S4=0.4628 S8=0.2591 S16=0.1690 S32=0.1812 S64=0.1659 S128=0.1802  | best S=64 (0.1659 ms)

FlashInfer rematch (split-K best-S vs flashinfer vs old single-block):
  D= 64 N=  1024 S= 32 | split-K 0.0328 ms | flashinfer 0.0461 ms | sk/fi 1.41x
  D= 64 N=  4096 S= 32 | split-K 0.0604 ms | flashinfer 0.0471 ms | sk/fi 0.78x
  D= 64 N= 16384 S= 32 | split-K 0.1792 ms | flashinfer 0.0584 ms | sk/fi 0.33x
  D=128 N=  1024 S= 32 | split-K 0.0379 ms | flashinfer 0.0481 ms | sk/fi 1.27x
  D=128 N=  4096 S= 32 | split-K 0.0963 ms | flashinfer 0.0543 ms | sk/fi 0.56x
  D=128 N= 16384 S= 32 | split-K 0.2867 ms | flashinfer 0.0881 ms | sk/fi 0.31x


In [ ]:
import torch, math, torch.nn.functional as F, flashinfer
dev="cuda"; torch.manual_seed(0)

def bench(fn, iters=100, warmup=30):
    warm=torch.randn(2048,2048,device=dev,dtype=torch.float16)
    for _ in range(30): _=warm@warm
    torch.cuda.synchronize()
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    s,e=torch.cuda.Event(enable_timing=True),torch.cuda.Event(enable_timing=True); ts=[]
    for _ in range(iters):
        s.record(); fn(); e.record(); torch.cuda.synchronize(); ts.append(s.elapsed_time(e))
    ts.sort(); return ts[len(ts)//2]

# find best S per shape, then compare at that S
def best_s_compare(N, D, H=32, Hkv=8, W=8):
    G=H//Hkv
    q_h=torch.randn(H,D,dtype=torch.float16,device=dev)
    k_kv=torch.randn(N,Hkv,D,dtype=torch.float16,device=dev)
    v_kv=torch.randn(N,Hkv,D,dtype=torch.float16,device=dev)
    kE=k_kv.transpose(0,1).repeat_interleave(G,0).contiguous()
    vE=v_kv.transpose(0,1).repeat_interleave(G,0).contiguous()
    qc=q_h.contiguous()
    # correctness at the S we'll report
    o_ref=F.scaled_dot_product_attention(q_h.view(H,1,1,D),kE.view(H,1,N,D),vE.view(H,1,N,D)).view(H,D)
    # pick best S by measurement
    best=(1e9,None)
    for S in [1,4,8,16,32,64,128]:
        o=mod_sk.decode_attn_splitk(qc,kE,vE,S,W)
        if not torch.allclose(o.float(),o_ref.float(),atol=2e-2):  # gate every S
            continue
        t=bench(lambda: mod_sk.decode_attn_splitk(qc,kE,vE,S,W))
        if t<best[0]: best=(t,S)
    t_sk,S_best=best
    t_fi=bench(lambda: flashinfer.single_decode_with_kv_cache(q_h,k_kv,v_kv))
    print(f"D={D:3d} N={N:6d} | split-K(S={S_best:3d}) {t_sk:.4f} ms | flashinfer {t_fi:.4f} ms | sk/fi {t_fi/t_sk:.2f}x")
    return dict(D=D,N=N,best_S=S_best,splitk_ms=t_sk,flashinfer_ms=t_fi,ratio=t_fi/t_sk)

import json
rows=[]
print("Final rematch — each shape at ITS best S, correctness-gated:")
for D in [64,128]:
    for N in [512,1024,2048,4096,8192,16384]:
        rows.append(best_s_compare(N,D))
import os; os.makedirs("benchmarks",exist_ok=True)
json.dump(dict(note="split-K decode vs FlashInfer single_decode; each shape at measured-best S; scratch alloc inside timed call (penalizes split-K)", rows=rows),
          open("benchmarks/flashinfer_compare.json","w"), indent=2)
print("\nsaved benchmarks/flashinfer_compare.json")

Final rematch — each shape at ITS best S, correctness-gated:
D= 64 N=   512 | split-K(S= 16) 0.0297 ms | flashinfer 0.0471 ms | sk/fi 1.59x
D= 64 N=  1024 | split-K(S=  8) 0.0317 ms | flashinfer 0.0471 ms | sk/fi 1.48x
D= 64 N=  2048 | split-K(S= 16) 0.0369 ms | flashinfer 0.0471 ms | sk/fi 1.28x
D= 64 N=  4096 | split-K(S= 16) 0.0573 ms | flashinfer 0.0471 ms | sk/fi 0.82x
D= 64 N=  8192 | split-K(S= 16) 0.0973 ms | flashinfer 0.0481 ms | sk/fi 0.49x
D= 64 N= 16384 | split-K(S= 64) 0.1659 ms | flashinfer 0.0584 ms | sk/fi 0.35x
D=128 N=   512 | split-K(S=  8) 0.0297 ms | flashinfer 0.0481 ms | sk/fi 1.62x
D=128 N=  1024 | split-K(S=  8) 0.0317 ms | flashinfer 0.0481 ms | sk/fi 1.52x
D=128 N=  2048 | split-K(S= 16) 0.0512 ms | flashinfer 0.0481 ms | sk/fi 0.94x
D=128 N=  4096 | split-K(S= 16) 0.0860 ms | flashinfer 0.0543 ms | sk/fi 0.63x
D=128 N=  8192 | split-K(S= 16) 0.1444 ms | flashinfer 0.0614 ms | sk/fi 0.43x
D=128 N= 16384 | split-K(S= 16) 0.2560 ms | flashinfer 0.0870 ms | sk/

In [ ]:
# Which vector width fits D over 32 lanes with every lane active?
for D in [64, 128]:
    print(f"D={D}:")
    for width in [2, 4, 8]:   # halfs per thread (32-bit, 64-bit, 128-bit load)
        total_threads_needed = D // width
        if D % width != 0:
            print(f"  width={width}: D not divisible — skip")
            continue
        lanes_used = total_threads_needed
        elems_per_lane_if_32 = D / 32 / 1  # for reference
        # can 32 lanes each own an equal vector chunk?
        if D % (32*width) == 0:
            vecs_per_lane = D // (32*width)
            print(f"  width={width}: 32 lanes each own {vecs_per_lane} vec(s) of {width} -> ALL lanes active, clean")
        else:
            print(f"  width={width}: only {lanes_used} lanes do loads ({lanes_used}/32 active) -> wastes {32-lanes_used} lanes")

D=64:
  width=2: 32 lanes each own 1 vec(s) of 2 -> ALL lanes active, clean
  width=4: only 16 lanes do loads (16/32 active) -> wastes 16 lanes
  width=8: only 8 lanes do loads (8/32 active) -> wastes 24 lanes
D=128:
  width=2: 32 lanes each own 2 vec(s) of 2 -> ALL lanes active, clean
  width=4: 32 lanes each own 1 vec(s) of 4 -> ALL lanes active, clean
  width=8: only 16 lanes do loads (16/32 active) -> wastes 16 lanes


In [ ]:
import os, math, torch, torch.nn.functional as F
from torch.utils.cpp_extension import load
os.makedirs("kernels", exist_ok=True)

splitk_vec_src = r'''
#include <torch/extension.h>
#include <c10/cuda/CUDAException.h>
#include <cuda_fp16.h>
#include <cuda_runtime.h>

// Vectorized split-K. Each lane owns VPT contiguous elements, loaded as one vector.
// D=64 -> VPT=2 (32-bit load); D=128 -> VPT=4 (64-bit load). All 32 lanes active.
// Lane t owns elements [t*VPT, t*VPT+VPT).  (contiguous, not strided)
template <int D, int W, int VPT>
__global__ void splitk_partial_vec_kernel(
    const __half* __restrict__ q, const __half* __restrict__ K, const __half* __restrict__ V,
    float* __restrict__ m_part, float* __restrict__ l_part, float* __restrict__ o_part,
    int N, int S, float scale)
{
    const int head=blockIdx.x, split=blockIdx.y;
    const int lane=threadIdx.x&31, warp=threadIdx.x>>5;
    const int chunk=(N+S-1)/S, lo=split*chunk, hi=min(lo+chunk,N);
    const __half* qh=q+(size_t)head*D;
    const __half* Kh=K+(size_t)head*N*D;
    const __half* Vh=V+(size_t)head*N*D;

    // load this lane's VPT contiguous q elements (vectorized) into registers
    float q_reg[VPT];
    const int off0 = lane*VPT;
    #pragma unroll
    for(int j=0;j<VPT;++j) q_reg[j]=__half2float(qh[off0+j]);

    float m_run=-INFINITY, l_run=0.0f, o_run[VPT];
    #pragma unroll
    for(int j=0;j<VPT;++j) o_run[j]=0.0f;

    for(int i=lo+warp; i<hi; i+=W){
        const __half* Ki=Kh+(size_t)i*D;
        // ---- vectorized load of VPT contiguous K elements for this lane ----
        float kk[VPT];
        if(VPT==2){
            float v = *reinterpret_cast<const float*>(Ki+off0);   // 32-bit = 2 halfs
            __half2 h = *reinterpret_cast<__half2*>(&v);
            kk[0]=__half2float(__low2half(h)); kk[1]=__half2float(__high2half(h));
        } else { // VPT==4, 64-bit load
            float2 v = *reinterpret_cast<const float2*>(Ki+off0);
            __half2 h0=*reinterpret_cast<__half2*>(&v.x), h1=*reinterpret_cast<__half2*>(&v.y);
            kk[0]=__half2float(__low2half(h0)); kk[1]=__half2float(__high2half(h0));
            kk[2]=__half2float(__low2half(h1)); kk[3]=__half2float(__high2half(h1));
        }
        float partial=0.0f;
        #pragma unroll
        for(int j=0;j<VPT;++j) partial+=q_reg[j]*kk[j];
        // full warp reduction (all 32 lanes contribute their VPT-element partials)
        #pragma unroll
        for(int o=16;o>0;o>>=1) partial+=__shfl_down_sync(0xffffffff,partial,o);
        float s=__shfl_sync(0xffffffff,partial,0)*scale;

        float m_new=fmaxf(m_run,s), beta=__expf(m_run-m_new), p=__expf(s-m_new);
        l_run=beta*l_run+p;
        const __half* Vi=Vh+(size_t)i*D;
        float vv[VPT];
        if(VPT==2){
            float v=*reinterpret_cast<const float*>(Vi+off0); __half2 h=*reinterpret_cast<__half2*>(&v);
            vv[0]=__half2float(__low2half(h)); vv[1]=__half2float(__high2half(h));
        } else {
            float2 v=*reinterpret_cast<const float2*>(Vi+off0);
            __half2 h0=*reinterpret_cast<__half2*>(&v.x), h1=*reinterpret_cast<__half2*>(&v.y);
            vv[0]=__half2float(__low2half(h0)); vv[1]=__half2float(__high2half(h0));
            vv[2]=__half2float(__low2half(h1)); vv[3]=__half2float(__high2half(h1));
        }
        #pragma unroll
        for(int j=0;j<VPT;++j) o_run[j]=beta*o_run[j]+p*vv[j];
        m_run=m_new;
    }

    // intra-block combine over W warps (guarded -inf, same as fixed scalar version)
    __shared__ float sm[W],sl[W],so[W][D];
    if(lane==0){sm[warp]=m_run;sl[warp]=l_run;}
    #pragma unroll
    for(int j=0;j<VPT;++j) so[warp][off0+j]=o_run[j];
    __syncthreads();
    if(warp==0){
        float m_c=-INFINITY,l_c=0.0f,o_c[VPT];
        #pragma unroll
        for(int j=0;j<VPT;++j) o_c[j]=0.0f;
        #pragma unroll
        for(int w=0;w<W;++w){ float mw=sm[w],mn=fmaxf(m_c,mw);
            float a=(mn==-INFINITY)?0.0f:__expf(m_c-mn);
            float b=(mn==-INFINITY)?0.0f:__expf(mw-mn); l_c=a*l_c+b*sl[w];
            #pragma unroll
            for(int j=0;j<VPT;++j) o_c[j]=a*o_c[j]+b*so[w][off0+j]; m_c=mn; }
        const size_t base=((size_t)head*S+split);
        if(lane==0){ m_part[base]=m_c; l_part[base]=l_c; }
        #pragma unroll
        for(int j=0;j<VPT;++j) o_part[base*D+off0+j]=o_c[j];
    }
}

template <int D>
__global__ void splitk_combine_kernel2(
    const float* __restrict__ m_part, const float* __restrict__ l_part,
    const float* __restrict__ o_part, __half* __restrict__ o, int S)
{
    const int head=blockIdx.x, lane=threadIdx.x&31, ELEMS=D/32;
    float m_c=-INFINITY,l_c=0.0f,o_c[ELEMS];
    #pragma unroll
    for(int e=0;e<ELEMS;++e) o_c[e]=0.0f;
    for(int s=0;s<S;++s){
        const size_t base=((size_t)head*S+s);
        float m_s=m_part[base], l_s=l_part[base];
        float mn=fmaxf(m_c,m_s);
        float a=(mn==-INFINITY)?0.0f:__expf(m_c-mn);
        float b=(mn==-INFINITY)?0.0f:__expf(m_s-mn);
        l_c=a*l_c+b*l_s;
        #pragma unroll
        for(int e=0;e<ELEMS;++e) o_c[e]=a*o_c[e]+b*o_part[base*D+lane+e*32];
        m_c=mn;
    }
    __half* oh=o+(size_t)head*D;
    #pragma unroll
    for(int e=0;e<ELEMS;++e) oh[lane+e*32]=__float2half(o_c[e]/l_c);
}

torch::Tensor decode_attn_splitk_vec(torch::Tensor q, torch::Tensor K, torch::Tensor V, int64_t S, int64_t W){
    const int H=K.size(0),N=K.size(1),D=K.size(2);
    TORCH_CHECK(D==64||D==128,"D");
    auto fopt=torch::TensorOptions().dtype(torch::kFloat32).device(q.device());
    auto m_part=torch::empty({H,S},fopt), l_part=torch::empty({H,S},fopt), o_part=torch::empty({H,S,D},fopt);
    auto o=torch::empty({H,D},q.options()); const float scale=1.0f/sqrtf((float)D);
    dim3 pgrid(H,S), pblock(32*W);
    #define LP(Dv,Wv,Vv) splitk_partial_vec_kernel<Dv,Wv,Vv><<<pgrid,pblock>>>( \
        (const __half*)q.data_ptr<at::Half>(),(const __half*)K.data_ptr<at::Half>(),(const __half*)V.data_ptr<at::Half>(), \
        m_part.data_ptr<float>(),l_part.data_ptr<float>(),o_part.data_ptr<float>(),N,(int)S,scale)
    if(D==64){switch(W){case 2:LP(64,2,2);break;case 4:LP(64,4,2);break;case 8:LP(64,8,2);break;default:TORCH_CHECK(false,"W");}}
    else{switch(W){case 2:LP(128,2,4);break;case 4:LP(128,4,4);break;case 8:LP(128,8,4);break;default:TORCH_CHECK(false,"W");}}
    #undef LP
    TORCH_CHECK(cudaGetLastError()==cudaSuccess,"partial");
    dim3 cgrid(H),cblock(32);
    if(D==64) splitk_combine_kernel2<64><<<cgrid,cblock>>>(m_part.data_ptr<float>(),l_part.data_ptr<float>(),o_part.data_ptr<float>(),(__half*)o.data_ptr<at::Half>(),(int)S);
    else      splitk_combine_kernel2<128><<<cgrid,cblock>>>(m_part.data_ptr<float>(),l_part.data_ptr<float>(),o_part.data_ptr<float>(),(__half*)o.data_ptr<at::Half>(),(int)S);
    TORCH_CHECK(cudaGetLastError()==cudaSuccess,"combine");
    return o;
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME,m){m.def("decode_attn_splitk_vec",&decode_attn_splitk_vec,"vectorized split-K");}
'''
open("kernels/decode_attn_splitk_vec.cu","w").write(splitk_vec_src)
mod_skv = load(name="decode_attn_splitk_vec", sources=["kernels/decode_attn_splitk_vec.cu"], extra_cuda_cflags=["-O3"], verbose=True)
print("vectorized split-K compiled OK")

vectorized split-K compiled OK


In [ ]:
dev="cuda"; torch.manual_seed(0)
def sdpa_ref(q,K,V):
    H,D=q.shape; N=K.shape[1]
    return F.scaled_dot_product_attention(q.view(H,1,1,D),K.view(H,1,N,D),V.view(H,1,N,D)).view(H,D)
def check_skv(H,N,D,S,W,atol=2e-2):
    q=torch.randn(H,D,dtype=torch.float16,device=dev).contiguous()
    K=torch.randn(H,N,D,dtype=torch.float16,device=dev).contiguous()
    V=torch.randn(H,N,D,dtype=torch.float16,device=dev).contiguous()
    out=mod_skv.decode_attn_splitk_vec(q,K,V,S,W); ref=sdpa_ref(q,K,V)
    d=(out-ref).abs().max().item(); ok=torch.allclose(out,ref,atol=atol)
    print(f"D={D:3d} H={H:3d} N={N:6d} S={S:4d} W={W} | diff {d:.3e} | {'PASS' if ok else 'FAIL'}")
    return ok
print("vectorized split-K correctness:")
ok=True
for D in [64,128]:
    for W in [4,8]:
        for (N,S) in [(1024,8),(4096,16),(16384,64)]:
            ok&=check_skv(32,N,D,S,W)
    ok&=check_skv(8,3,D,8,4)    # empty splits
    ok&=check_skv(8,1,D,4,2)    # N=1
    ok&=check_skv(8,100,D,7,4)  # N not div by S
print("\nALL PASS" if ok else "\nSOME FAILED — stop"); assert ok

vectorized split-K correctness:
D= 64 H= 32 N=  1024 S=   8 W=4 | diff 1.221e-04 | PASS
D= 64 H= 32 N=  4096 S=  16 W=4 | diff 6.104e-05 | PASS
D= 64 H= 32 N= 16384 S=  64 W=4 | diff 3.052e-05 | PASS
D= 64 H= 32 N=  1024 S=   8 W=8 | diff 1.221e-04 | PASS
D= 64 H= 32 N=  4096 S=  16 W=8 | diff 6.104e-05 | PASS
D= 64 H= 32 N= 16384 S=  64 W=8 | diff 3.052e-05 | PASS
D= 64 H=  8 N=     3 S=   8 W=4 | diff 1.953e-03 | PASS
D= 64 H=  8 N=     1 S=   4 W=2 | diff 0.000e+00 | PASS
D= 64 H=  8 N=   100 S=   7 W=4 | diff 2.441e-04 | PASS
D=128 H= 32 N=  1024 S=   8 W=4 | diff 1.221e-04 | PASS
D=128 H= 32 N=  4096 S=  16 W=4 | diff 6.104e-05 | PASS
D=128 H= 32 N= 16384 S=  64 W=4 | diff 3.052e-05 | PASS
D=128 H= 32 N=  1024 S=   8 W=8 | diff 1.221e-04 | PASS
D=128 H= 32 N=  4096 S=  16 W=8 | diff 6.104e-05 | PASS
D=128 H= 32 N= 16384 S=  64 W=8 | diff 3.052e-05 | PASS
D=128 H=  8 N=     3 S=   8 W=4 | diff 9.766e-04 | PASS
D=128 H=  8 N=     1 S=   4 W=2 | diff 0.000e+00 | PASS
D=128 H=  8 N=  

In [ ]:
import torch, math, torch.nn.functional as F, flashinfer, json, os
dev="cuda"; torch.manual_seed(0)

def bench(fn, iters=100, warmup=30):
    warm=torch.randn(2048,2048,device=dev,dtype=torch.float16)
    for _ in range(30): _=warm@warm
    torch.cuda.synchronize()
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    s,e=torch.cuda.Event(enable_timing=True),torch.cuda.Event(enable_timing=True); ts=[]
    for _ in range(iters):
        s.record(); fn(); e.record(); torch.cuda.synchronize(); ts.append(s.elapsed_time(e))
    ts.sort(); return ts[len(ts)//2]

def best_t(kernel_fn, qc,kE,vE, Ws=(4,8), Ss=(1,4,8,16,32,64,128)):
    best=(1e9,None,None)
    for W in Ws:
        for S in Ss:
            t=bench(lambda: kernel_fn(qc,kE,vE,S,W))
            if t<best[0]: best=(t,S,W)
    return best

rows=[]
print("vectorized vs scalar split-K vs FlashInfer (each at its best S,W):")
for D in [64,128]:
    for N in [512,1024,2048,4096,8192,16384]:
        H,Hkv=32,8; G=H//Hkv
        q_h=torch.randn(H,D,dtype=torch.float16,device=dev)
        k_kv=torch.randn(N,Hkv,D,dtype=torch.float16,device=dev)
        v_kv=torch.randn(N,Hkv,D,dtype=torch.float16,device=dev)
        kE=k_kv.transpose(0,1).repeat_interleave(G,0).contiguous()
        vE=v_kv.transpose(0,1).repeat_interleave(G,0).contiguous()
        qc=q_h.contiguous()
        # correctness gate on both kernels before timing
        o_ref=F.scaled_dot_product_attention(q_h.view(H,1,1,D),kE.view(H,1,N,D),vE.view(H,1,N,D)).view(H,D)
        o_sc=mod_sk.decode_attn_splitk(qc,kE,vE,16,8)
        o_v =mod_skv.decode_attn_splitk_vec(qc,kE,vE,16,8)
        assert torch.allclose(o_sc.float(),o_ref.float(),atol=2e-2), f"scalar FAIL D={D} N={N}"
        assert torch.allclose(o_v.float(), o_ref.float(),atol=2e-2), f"vec FAIL D={D} N={N}"
        t_sc,Ss,Ws=best_t(mod_sk.decode_attn_splitk, qc,kE,vE)
        t_v ,Sv,Wv=best_t(mod_skv.decode_attn_splitk_vec, qc,kE,vE)
        t_fi=bench(lambda: flashinfer.single_decode_with_kv_cache(q_h,k_kv,v_kv))
        print(f"D={D:3d} N={N:6d} | scalar {t_sc:.4f}(S{Ss}) | vec {t_v:.4f}(S{Sv}) | fi {t_fi:.4f} "
              f"| vec/scalar {t_sc/t_v:.2f}x | vec/fi {t_fi/t_v:.2f}x")
        rows.append(dict(D=D,N=N,scalar_ms=t_sc,vec_ms=t_v,fi_ms=t_fi,
                         vec_vs_scalar=t_sc/t_v, vec_vs_fi=t_fi/t_v, best_S_vec=Sv, best_W_vec=Wv))
json.dump(dict(note="vectorized vs scalar split-K vs FlashInfer; best S,W per shape; scratch alloc in timed call",
               rows=rows), open("benchmarks/vectorized_compare.json","w"), indent=2)
print("\nsaved benchmarks/vectorized_compare.json")

vectorized vs scalar split-K vs FlashInfer (each at its best S,W):
D= 64 N=   512 | scalar 0.0297(S16) | vec 0.0297(S16) | fi 0.0471 | vec/scalar 1.00x | vec/fi 1.59x
D= 64 N=  1024 | scalar 0.0297(S32) | vec 0.0297(S32) | fi 0.0471 | vec/scalar 1.00x | vec/fi 1.59x
D= 64 N=  2048 | scalar 0.0307(S16) | vec 0.0307(S16) | fi 0.0461 | vec/scalar 1.00x | vec/fi 1.50x
D= 64 N=  4096 | scalar 0.0563(S16) | vec 0.0553(S16) | fi 0.0471 | vec/scalar 1.02x | vec/fi 0.85x
D= 64 N=  8192 | scalar 0.0973(S16) | vec 0.0952(S16) | fi 0.0471 | vec/scalar 1.02x | vec/fi 0.49x
D= 64 N= 16384 | scalar 0.1669(S64) | vec 0.1618(S64) | fi 0.0584 | vec/scalar 1.03x | vec/fi 0.36x
D=128 N=   512 | scalar 0.0297(S16) | vec 0.0297(S8) | fi 0.0481 | vec/scalar 1.00x | vec/fi 1.62x
D=128 N=  1024 | scalar 0.0317(S8) | vec 0.0307(S8) | fi 0.0481 | vec/scalar 1.03x | vec/fi 1.57x
D=128 N=  2048 | scalar 0.0512(S16) | vec 0.0502(S16) | fi 0.0481 | vec/scalar 1.02x | vec/fi 0.96x
D=128 N=  4096 | scalar 0.0860(S16) 

In [ ]:
%%writefile profile_splitk.py
import torch, math
from torch.utils.cpp_extension import load
mod = load(name="decode_attn_splitk", sources=["kernels/decode_attn_splitk.cu"],
           extra_cuda_cflags=["-O3"], verbose=False)
dev="cuda"; torch.manual_seed(0)
H,N,D,S,W = 32, 16384, 128, 16, 8          # the long-context shape where we're 0.35x of FlashInfer
q=torch.randn(H,D,dtype=torch.float16).to(dev).contiguous()
K=torch.randn(H,N,D,dtype=torch.float16).to(dev).contiguous()
V=torch.randn(H,N,D,dtype=torch.float16).to(dev).contiguous()
for _ in range(20): _=mod.decode_attn_splitk(q,K,V,S,W)
torch.cuda.synchronize()
_=mod.decode_attn_splitk(q,K,V,S,W); torch.cuda.synchronize()
print("done")

Writing profile_splitk.py


In [ ]:
# profile BOTH kernels split-K launches: the partial kernel AND the combine kernel.
# --launch-count 2 so we catch both; grep shows which is which.
!ncu --target-processes all --kernel-name regex:splitk --launch-count 2 \
     --section SpeedOfLight --section Occupancy --section WarpStateStats \
     python profile_splitk.py 2>&1 | grep -Ei \
     "splitk_partial|splitk_combine|Duration|Memory Throughput|DRAM Throughput|Compute \(SM\)|Achieved Occupancy|Achieved Active Warps|Long Scoreboard|Warp Cycles Per Issued" | head -n 60

==PROF== Profiling "splitk_partial_kernel": 0%....50%....100% - 12 passes
==PROF== Profiling "splitk_combine_kernel": 0%....50%....100% - 12 passes
  void splitk_partial_kernel<128, 8>(const __half *, const __half *, const __half *, float *, float *, float *, int, int, float) (32, 16, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 8.0
    Memory Throughput                 %        69.37
    DRAM Throughput                   %        69.37
    Duration                         us       251.07
    Compute (SM) Throughput           %        29.22
    Warp Cycles Per Issued Instruction             cycle        30.35
    Achieved Occupancy                        %        57.27
    Achieved Active Warps Per SM           warp        36.65
          The difference between calculated theoretical (100.0%) and measured achieved occupancy (57.3%) can be the     
  void splitk_combine_kernel<128>(const float *, const float *, const float *, __half *, int) (32, 1, 1)x(32, 1, 1), Context 1, Stream 

In [ ]:
import torch, torch.nn.functional as F
dev="cuda"; torch.manual_seed(0)
def bench(fn, iters=100, warmup=30):
    warm=torch.randn(2048,2048,device=dev,dtype=torch.float16)
    for _ in range(30): _=warm@warm
    torch.cuda.synchronize()
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    s,e=torch.cuda.Event(enable_timing=True),torch.cuda.Event(enable_timing=True); ts=[]
    for _ in range(iters):
        s.record(); fn(); e.record(); torch.cuda.synchronize(); ts.append(s.elapsed_time(e))
    ts.sort(); return ts[len(ts)//2]

print("W x S sweep at D=128 N=16384 (the shape where we're 0.35x of FlashInfer):")
H,Hkv,D,N=32,8,128,16384; G=H//Hkv
q=torch.randn(H,D,dtype=torch.float16,device=dev).contiguous()
k_kv=torch.randn(N,Hkv,D,dtype=torch.float16,device=dev)
kE=k_kv.transpose(0,1).repeat_interleave(G,0).contiguous()
vE=torch.randn(N,Hkv,D,dtype=torch.float16,device=dev).transpose(0,1).repeat_interleave(G,0).contiguous()
# correctness guard once
o_ref=F.scaled_dot_product_attention(q.view(H,1,1,D),kE.view(H,1,N,D),vE.view(H,1,N,D)).view(H,D)
best=(1e9,None,None)
for W in [2,4,8]:
    row=f"  W={W}: "
    for S in [16,32,64,128,256]:
        try:
            o=mod_sk.decode_attn_splitk(q,kE,vE,S,W)
            if not torch.allclose(o.float(),o_ref.float(),atol=2e-2):
                row+=f"S{S}=BAD "; continue
            t=bench(lambda: mod_sk.decode_attn_splitk(q,kE,vE,S,W))
            row+=f"S{S}={t:.4f} "
            if t<best[0]: best=(t,S,W)
        except Exception as ex:
            row+=f"S{S}=ERR "
    print(row)
import flashinfer
t_fi=bench(lambda: flashinfer.single_decode_with_kv_cache(q,k_kv, torch.randn(N,Hkv,D,dtype=torch.float16,device=dev)))
print(f"\n  best split-K: {best[0]:.4f} ms at S={best[1]} W={best[2]} | FlashInfer {t_fi:.4f} ms | ratio {t_fi/best[0]:.2f}x")
print(f"  (previous best at this shape was ~0.256 ms, 0.34x)")

W x S sweep at D=128 N=16384 (the shape where we're 0.35x of FlashInfer):
  W=2: S16=0.7465 S32=0.3656 S64=0.2765 S128=0.3144 S256=0.3932 
  W=4: S16=0.3615 S32=0.2632 S64=0.3062 S128=0.3092 S256=0.3748 
  W=8: S16=0.2560 S32=0.2847 S64=0.2693 S128=0.2970 S256=0.3789 

  best split-K: 0.2560 ms at S=16 W=8 | FlashInfer 0.1608 ms | ratio 0.63x
  (previous best at this shape was ~0.256 ms, 0.34x)


In [ ]:
import os, json, glob
os.makedirs("benchmarks", exist_ok=True)
print("benchmark files on disk:")
for f in sorted(glob.glob("benchmarks/*.json")):
    print(" ", f, f"({os.path.getsize(f)} bytes)")
# consolidate everything measured tonight so nothing is lost on disconnect
results={os.path.splitext(os.path.basename(f))[0]: json.load(open(f)) for f in glob.glob("benchmarks/*.json")}
json.dump(results, open("results.json","w"), indent=2)
print("\nconsolidated into results.json:", list(results.keys()))

benchmark files on disk:
  benchmarks/flashinfer_compare.json (2357 bytes)
  benchmarks/vectorized_compare.json (3502 bytes)

consolidated into results.json: ['vectorized_compare', 'flashinfer_compare']
